# Group Recommendations Demo

This notebook demonstrates how to generate recommendations for groups of users using different aggregation strategies.

## Features Demonstrated:
- Multiple group recommendation strategies
- Group preference analysis
- Strategy comparison
- Recommendation explanations

In [ ]:
# Import required libraries
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
from src.group_recommendations import GroupRecommendationEngine, compare_group_strategies
from src.recommendations import load_user_data_with_tmdb
from src.data_processing import load_movielens_data
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

## 1. Load Data and Model

In [ ]:
# Load MovieLens data
_, ratings_df, movies_df, links_df = load_movielens_data(
    "../ml-latest-small/ratings.csv",
    "../ml-latest-small/movies.csv", 
    "../ml-latest-small/links.csv"
)

print(f"Loaded {len(movies_df)} movies and {len(ratings_df)} ratings")
print(f"Rating range: {ratings_df['rating'].min()} - {ratings_df['rating'].max()}")

In [ ]:
# Initialize group recommendation engine
engine = GroupRecommendationEngine("../models/svdpp.pkl")
print("Group recommendation engine initialized!")

## 2. Create Sample Group

Let's create a group with different user preferences to demonstrate the system.

In [ ]:
# Create sample group with different preferences
# In a real scenario, you would load actual user data

# User 1: Likes action and sci-fi
user1_ratings = {
    '1': 5.0,    # Toy Story
    '2': 4.0,    # Jumanji
    '3': 4.5,    # Grumpier Old Men
    '6': 5.0,    # Heat
    '47': 4.5,   # Seven
    '50': 5.0,   # Usual Suspects
    '32': 4.0,   # Twelve Monkeys
}

# User 2: Likes romance and drama
user2_ratings = {
    '1': 3.5,    # Toy Story
    '3': 3.0,    # Grumpier Old Men
    '10': 4.5,   # GoldenEye
    '16': 5.0,   # Casino
    '25': 4.0,   # Leaving Las Vegas
    '32': 3.5,   # Twelve Monkeys
    '39': 4.5,   # Clueless
}

# User 3: Likes comedy and family films
user3_ratings = {
    '1': 5.0,    # Toy Story
    '2': 4.5,    # Jumanji
    '3': 4.0,    # Grumpier Old Men
    '31': 4.5,   # Dangerous Minds
    '47': 3.5,   # Seven
    '52': 4.0,   # Mighty Aphrodite
    '39': 5.0,   # Clueless
}

# Combine into group
group_ratings = {
    'Alice': user1_ratings,
    'Bob': user2_ratings,
    'Carol': user3_ratings
}

print("Sample group created:")
for user, ratings in group_ratings.items():
    print(f"  {user}: {len(ratings)} ratings")

## 3. Analyze Group Preferences

In [ ]:
# Analyze group compatibility and preferences
analysis = engine.analyze_group_preferences(group_ratings, movies_df)

print("Group Analysis:")
print(f"  Number of users: {analysis['num_users']}")
print(f"  Total ratings: {analysis['total_ratings']}")
print(f"  Average ratings per user: {analysis['avg_ratings_per_user']:.1f}")
print(f"  Average group similarity: {analysis['avg_group_similarity']:.3f}")
print(f"  Common movies: {analysis['common_movies_count']}")

print("\nUser Similarities:")
for pair, similarity in analysis['user_similarities'].items():
    print(f"  {pair}: {similarity:.3f}")

print("\nTop Consensus Movies:")
for movie, rating, std in analysis['consensus_movies'][:5]:
    print(f"  {movie}: {rating:.1f}★ (±{std:.2f})")

if analysis['disagreement_movies']:
    print("\nTop Disagreement Movies:")
    for movie, rating, std in analysis['disagreement_movies'][:3]:
        print(f"  {movie}: {rating:.1f}★ (±{std:.2f})")

## 4. Generate Group Recommendations

Let's try different aggregation strategies and see how they differ.

In [ ]:
# Strategy 1: Average satisfaction
print("=== AVERAGE STRATEGY ===")
avg_recs = engine.get_group_recommendations(
    group_ratings, movies_df, strategy="average", top_n=10
)

for i, (title, score, individual) in enumerate(avg_recs, 1):
    print(f"{i:2d}. {title} — {score}★")
    individual_str = ", ".join([f"{user}: {score}★" for user, score in individual.items() if score > 0])
    print(f"     Individual: {individual_str}")

In [ ]:
# Strategy 2: Least misery (ensure no one is disappointed)
print("\n=== LEAST MISERY STRATEGY ===")
lm_recs = engine.get_group_recommendations(
    group_ratings, movies_df, strategy="least_misery", top_n=10
)

for i, (title, score, individual) in enumerate(lm_recs, 1):
    print(f"{i:2d}. {title} — {score}★")
    individual_str = ", ".join([f"{user}: {score}★" for user, score in individual.items() if score > 0])
    print(f"     Individual: {individual_str}")

In [ ]:
# Strategy 3: Consensus (balance coverage and agreement)
print("\n=== CONSENSUS STRATEGY ===")
consensus_recs = engine.get_group_recommendations(
    group_ratings, movies_df, strategy="consensus", top_n=10
)

for i, (title, score, individual) in enumerate(consensus_recs, 1):
    print(f"{i:2d}. {title} — {score}★")
    individual_str = ", ".join([f"{user}: {score}★" for user, score in individual.items() if score > 0])
    print(f"     Individual: {individual_str}")

## 5. Get Recommendations with Explanations

In [ ]:
# Get detailed recommendations with explanations
detailed_results = engine.get_group_recommendations_with_explanation(
    group_ratings, movies_df, strategy="hybrid", top_n=5
)

print("=== HYBRID STRATEGY WITH EXPLANATIONS ===")
for i, ((title, score, individual), explanation) in enumerate(
    zip(detailed_results['recommendations'], detailed_results['explanations']), 1
):
    print(f"\n{i}. {title} — {score}★")
    print(f"   {explanation}")
    individual_str = ", ".join([f"{user}: {score}★" for user, score in individual.items() if score > 0])
    print(f"   Individual scores: {individual_str}")

## 6. Compare All Strategies

In [ ]:
# Compare all strategies side by side
comparison_df = compare_group_strategies(
    group_ratings, movies_df, "../models/svdpp.pkl", top_n=10
)

print("Strategy Comparison:")
print(comparison_df.to_string(index=False))

## 7. Test with Real User Data

If you have actual Letterboxd data, you can use it here.

In [ ]:
# Example with real user data (uncomment if you have the data)
# try:
#     # Load real user data
#     alex_ratings = load_user_data_with_tmdb(
#         "../alex_data/ratings_with_tmdb.csv",
#         "../ml-latest-small/links.csv"
#     )
#     
#     # Create a group with Alex and synthetic users
#     real_group = {
#         'Alex': alex_ratings,
#         'Friend1': user1_ratings,
#         'Friend2': user2_ratings
#     }
#     
#     # Generate recommendations
#     real_recs = engine.get_group_recommendations(
#         real_group, movies_df, strategy="consensus", top_n=5
#     )
#     
#     print("Real Group Recommendations:")
#     for i, (title, score, individual) in enumerate(real_recs, 1):
#         print(f"{i}. {title} — {score}★")
#         
# except FileNotFoundError:
#     print("Real user data not found. Using synthetic data only.")

print("Demo completed! Try modifying the user preferences to see how recommendations change.")

## Summary

This demo showcased:

1. **Group Preference Analysis**: Understanding compatibility between group members
2. **Multiple Strategies**: 
   - **Average**: Maximizes overall group satisfaction
   - **Least Misery**: Ensures no one is disappointed
   - **Most Pleasure**: Maximizes peak satisfaction
   - **Consensus**: Balances coverage and agreement
   - **Hybrid**: Combines satisfaction with fairness
3. **Explanations**: Human-readable reasoning for recommendations
4. **Strategy Comparison**: Quantitative analysis of different approaches

## Next Steps

- Try different group compositions
- Experiment with fairness weights in hybrid strategy
- Test with real user data from Letterboxd exports
- Implement additional strategies (e.g., Borda count, approval voting)